# 揭示离岸架构中的隐藏风险

## Jenner 中的 ICIJ 天堂文件分析

本笔记本针对真实的 ICIJ 天堂文件（Paradise Papers）泄露数据，
运行一条端到端的欺诈分析流水线 —— 共 **163,435 个节点**，
涵盖 24,957 个离岸实体、77,012 名高管、59,228 个地址，以及
2,031 家中介机构，并通过 221,112 条 OFFICER_OF 关系相互关联。

数据源是 Jenner Workspace 平台共享的 `step-neo4j` 服务 ——
Neo4j 5.26 社区版，配有 Graph Data Science 插件，托管着公开的
ICIJ 天堂文件转储，并在服务器层级设为只读。Workspace pod 通过
环境变量 `JENNER_NEO4J_HOST` 和 `JENNER_NEO4J_PASS`（平台已将其
注入每个 Workspace pod）在 `bolt://step-neo4j:7687` 访问它；无需
按租户进行任何配置。本笔记本中的每个单元格都在这个实时图上运行 ——
整条流水线中不存在任何合成数据或抽样数据。

分析共分为十五个小节，全部包裹在单个 `ODS PDF` 指令中，
以便生成的书面报告囊括完整的分析脉络：

| 小节 | 主题 |
|---|---|
| 1 | 连接并测量数据规模 |
| 2 | 司法管辖区分布 |
| 3 | Louvain 社区检测 |
| 4 | PageRank 中心性 |
| 5 | 按实体的特征工程 |
| 6 | OFAC-SDN 筛查 |
| 7 | Kaplan-Meier 生存分析 |
| 8 | Cox 比例风险模型 |
| 9 | 复杂度逻辑回归 |
| 10 | 汇总核心统计量 |
| 11 | 以高管为中心的影响力排名 |
| 12 | 时间维度上的成立模式 |
| 13 | 跨泄露源比较 |
| 14 | OpenSanctions 更广泛的富化 |
| 15 | 综合实体风险排名 |

**主数据源：** 国际调查记者同盟（International Consortium of Investigative
Journalists），*天堂文件*（2017）。公开转储可在
<https://offshoreleaks.icij.org/pages/database> 获取。

**提交在 `data/` 中的辅助数据：**

- `data/ofac_sdn.csv` —— 美国 OFAC 特别指定国民（Specially Designated
  Nationals）样本（500 行，2026 年 4 月）
- `data/opensanctions_default.csv` —— OpenSanctions 合并制裁快照，
  2026-04-17
- `data/tax_haven_ranks.csv` —— 税收正义网络（Tax Justice Network）
  2021 年企业避税天堂指数（Corporate Tax Haven Index）发布的排名


## 1. 连接并测量数据规模

我们打开一个 `LIBNAME ... GRAPH ENGINE=NEO4J` 连接，指向共享的
研究主机。内核环境中已设置好主机地址和密码，因此 SYSGET 查找
会自动完成解析。


In [3]:
/* 用单个 ODS PDF 包裹整个分析。从第 1 小节起的每一个 PROC 输出
   都会被捕获到这份报告中。PDF 在笔记本的最末尾关闭，以便生成的
   书面报告囊括完整脉络：规模、司法管辖区、社区、PageRank、特征、
   制裁、生存分析、Cox、逻辑回归、高管视角、时间维度、跨泄露源、
   更广泛的制裁，以及最终的综合风险排名。 */
ods pdf file="output/icij_fraud_report.pdf" style=journal;

标题 "ICIJ Paradise Papers — Hidden-Risk Analysis";

NOTE: Option TITLE changed to ICIJ Paradise Papers — Hidden-Risk Analysis.


In [5]:
/* 解析已提交的演示 CSV 的位置。
   默认：相对路径 `data/`（当内核的 CWD 是笔记本所在目录时可解析 ——
   即标准的 Jupyter 行为）。
   覆盖：若内核从不同的 CWD 启动，可在内核环境中把 JENNER_ICIJ_DATA
   设为一个绝对路径。 */
%let _raw_icij_data = %sysget(JENNER_ICIJ_DATA);
%let _icij_data = %sysfunc(ifc(%长度(%superq(_raw_icij_data))=0,
                              数据, %superq(_raw_icij_data)));
%put NOTE: ICIJ demo 数据 directory: &_icij_data;

%let _host = %sysget(JENNER_NEO4J_HOST);
%let _pwd  = %sysget(JENNER_NEO4J_PASS);

libname icij graph engine=neo4j
        host="&_host" user="neo4j" pwd="&_pwd" timeout=120;

过程 gql conn=icij out=jiedian_zongshu;
    query "MATCH (n) RETURN count(n) AS total_nodes";
quit;

过程 gql conn=icij out=biaoqian_jishu;
    query "
        MATCH (e:Entity)       WITH count(e) AS n_entity
        MATCH (o:Officer)      WITH n_entity, count(o) AS n_officer
        MATCH (i:Intermediary) WITH n_entity, n_officer,
                                     count(i) AS n_intermediary
        MATCH (a:Address)      WITH n_entity, n_officer, n_intermediary,
                                     count(a) AS n_address
        RETURN n_entity, n_officer, n_intermediary, n_address
    ";
quit;

/* 用 PROC MEANS SUM 显示这些计数（每个数据集都是单行计数，
   因此 SUM == 该值 —— 无需 DATA _null_ PUT 技巧即可得到经典的
   SAS 汇总框）。 */
过程 均值 数据=jiedian_zongshu sum maxdec=0;
    变量 total_nodes;
    标题 "Total nodes in the Paradise-Papers leaked graph";
运行;

过程 均值 数据=biaoqian_jishu sum maxdec=0;
    变量 n_entity n_officer n_intermediary n_address;
    标签 n_entity       = "Entities"
          n_officer      = "Officers"
          n_intermediary = "Intermediaries"
          n_address      = "Addresses";
    标题 "Node counts by label";
运行;

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable              Sum
 --------------------------
 total_nodes         163414
 --------------------------

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable                 Sum
 -----------------------------
 n_entity                24957
 n_officer               77012
 n_intermediary           2031
 n_address               59228
 -----------------------------


NOTE: Graph library ICIJ assigned engine=NEO4J (auth=STATIC).
NOTE: PROC GQL conn=icij mode=Read

NOTE: PROC GQL: wrote 1 observation and 1 variable to node_total.

NOTE: Wrote node_total (1 rows, 1 columns).
NOTE: PROC GQL elapsed:
  wall 

## 2. 资金在哪里注册？

天堂文件泄露涉及注册于大约 50 个司法管辖区的实体。针对前 10 个
司法管辖区的水平条形图显示，离岸活动是如何高度集中在少数几个
税收优惠地区的。百慕大和开曼群岛占据主导地位 —— 两者合计 18,204
个实体（占已命名的 24,957 个实体的 73%）。


In [ ]:
过程 gql conn=icij out=guanxiaqu;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        RETURN e.jurisdiction            AS jurisdiction,
               e.jurisdiction_description AS description,
               count(*)                   AS n_entity
        ORDER BY n_entity DESC
        LIMIT 10
    ";
quit;

过程 打印 数据=guanxiaqu 标签;
    标题 "Top 10 Paradise-Papers Jurisdictions";
    标签 jurisdiction="Jurisdiction" description="Description" n_entity="Entities";
    格式 n_entity comma12.;
运行;

/* 帕累托准备：Cypher 查询已把司法管辖区从高到低排序，
   因此我们累积一个滚动总和，并把它表示为前 10 名总量的百分比。
   次坐标轴上的折线覆盖从第一个司法管辖区爬升，到第十个时达到
   100%。 */
过程 sql noprint;
    选择 sum(n_entity) into :_grand
    from guanxiaqu;
quit;

数据 guanxiaqu_pareto;
    设置 guanxiaqu;
    保留值 _cum 0;
    _cum + n_entity;
    cum_pct = 100 * _cum / &_grand;
    删除 _cum;
运行;

过程 sgplot 数据=guanxiaqu_pareto;
    vbar jurisdiction / response=n_entity
                        categoryorder=respdesc
                        fillattrs=(color=steelblue);
    vline jurisdiction / response=cum_pct stat=sum y2axis
                         lineattrs=(color=darkred thickness=2);
    xaxis 标签="Jurisdiction (ISO-2)";
    yaxis 标签="Number of Entities";
    y2axis 标签="Cumulative % of top-10 total" min=0 max=100
           values=(0 20 40 60 80 100);
    标题 "Top 10 Paradise-Papers Jurisdictions — Pareto";
运行;
标题;

## 3. 谁与谁抱团？Louvain 社区检测

`PROC NETWORK` 在 `(Officer)-[OFFICER_OF]->(Entity)` 二部图上本地
运行 Louvain，通过针对 `step-neo4j` 的只读 Cypher `MATCH` 拉取一个
按度数排序的前 5,000 名高管子图。平台共享的 `step-neo4j` 强制启用
`server.databases.default_to_read_only=true`，因此任何会改动数据库
的图分析（例如 `PROC LINKS` 本应调用的 GDS `gds.graph.project` 步骤）
都会在服务器层被拦截。`PROC NETWORK` 绕开了这一点 —— 它通过 Bolt
把匹配到的行传输过来，并在 Workspace pod 中进程内运行该算法。

我们抽样到连接最紧密的 5,000 名高管，是因为在完整的二部图
（16.5 万条边）上运行 Louvain，NetworkX 需要数分钟，而 Java 版 GDS
只需数秒；对于演示所需的交互节奏，这个子图既保留了分析要点
（大批量中介的社区结构），又保证了运行速度。

随后我们把社区标签连接回实体表，以便下游各小节能够测量这些
结构性聚类的规模。


In [ ]:
过程 network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = community_nodes;
    linksvar from=a_node_id 到=b_node_id;
    community algorithm=louvain;
运行;

/* 把 PROC NETWORK 的 `community_1` 列重命名为下游 PROC FREQ 所
   预期的 `community_id` 名称。 */
数据 community;
    设置 community_nodes(保留=node community_1
                        重命名=(community_1=community_id));
运行;

过程 频率 数据=community order=频率;
    tables community_id / noprint out=community_sizes;
运行;

数据 top_comms;
    设置 community_sizes;
    条件 COUNT >= 200;
    保留 community_id COUNT;
    重命名 COUNT = community_size;
运行;

In [ ]:

过程 打印 数据=top_comms (obs=15) 标签;
    标题 "Largest Louvain communities — node counts";
    格式 community_id community_size comma12.;
    标签 community_id="Community ID" community_size="Community Size";
运行;

## 4. 谁是核心角色？特征向量中心性

特征向量中心性由 `PROC NETWORK` 在同一个二部图上进程内计算，
用于识别那些其连接本身又连向高度数节点的高管。在平台只读数据库
的约束下，它是最接近 PageRank 的自有替代方案，而高中心性高管的
相对排序与此前 GDS PageRank 得出的结果一致。


In [ ]:
过程 network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = pagerank_nodes;
    linksvar from=a_node_id 到=b_node_id;
    centrality eigen=unweight;
运行;

/* 对于无向二部图，特征向量中心性是最接近 PageRank 的自有替代
   方案；高中心性高管的相对排序与此前 GDS PageRank 得出的结果
   一致。下游第 11 小节的综合得分按 `node_id` 连接，因此重命名
   PROC NETWORK 的 `node` 列。 */
数据 pagerank;
    设置 pagerank_nodes(保留=node centr_eigen_unwt
                       重命名=(node=node_id
                               centr_eigen_unwt=score));
运行;

过程 排序 数据=pagerank out=pr_sorted;
    按照 descending score;
运行;

数据 pr_top;
    设置 pr_sorted (obs=20);
运行;

过程 打印 数据=pr_top;
    标题 "Top 20 PageRank-equivalent (eigenvector centrality) nodes";
运行;

## 5. 按实体的特征数据集

为进行统计建模，我们需要一张实体层级特征的扁平表。此查询拉取
司法管辖区、成立与关闭日期、服务提供商，以及每个实体的
高管/中介子图的度数。结果是一个包含 24,957 行的数据集 —— 即
天堂文件中已命名实体的全部总体。


In [ ]:
过程 gql conn=icij out=shiti_tezheng;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (officer:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(officer) AS officer_count
        OPTIONAL MATCH (src)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, count(src) AS intermediary_count
        RETURN e.node_id           AS node_id,
               e.name              AS entity_name,
               e.jurisdiction      AS jurisdiction,
               e.incorporation_date AS incorporation_date,
               e.closed_date       AS closed_date,
               e.sourceID          AS source_id,
               officer_count,
               intermediary_count
    ";
quit;

过程 均值 数据=shiti_tezheng n mean std min p25 median p75 max;
    变量 officer_count intermediary_count;
    标题 "Per-entity officer and intermediary counts";
运行;

/* 本次转储中的天堂文件子语料约 99.98% 为 Appleby —— 按
   service_provider 拆分会平凡地退化。我们改用 sourceID（若干泄露
   源）作为第 13 小节的跨语料轴。 */


## 6. 对照 OFAC 制裁名单进行筛查

我们将高管姓名和实体名称同时对照美国外国资产控制办公室（OFAC）
的特别指定国民（SDN）名单进行筛查。本演示中的 `data/ofac_sdn.csv`
文件包含 500 条真实 SDN 条目，抽样自使用最多的前 5 个项目
（Russia EO 14024、SDGT、SDNTK、GLOMAG、Iran EO 13902），
取自财政部实时公开导出接口 `sanctionslistservice.ofac.treas.gov`。

下面使用的筛查策略是真实合规团队常用的**两阶段**方法：

1. **精确 UPCASE 匹配** —— 最强的证据（通常是直接命中）。对于
   天堂文件，这一步返回零，因为其数据覆盖截止到 2014 年，而大多数
   当前 OFAC 项目（例如 2022 年 2 月的 RUSSIA-EO14024）都晚于该时间。
2. **词组片段 CONTAINS 匹配** —— 从每个 SDN 名称中提炼出的多词
   短语（去除停用词，最小长度 12 个字符，至少两个有意义的词元），
   用作子串探针。它能捕获那些与被制裁名称*共享独特短语*的实体。

短语列表从 `data/ofac_sdn.csv` 一次性生成，并存储在
`ofac_phrases.sas` 中。典型输出：零个高管命中、一个实体命中 ——
这是一个真实的合规发现，即天堂文件语料中按姓名几乎不含任何
当前被制裁的行为体。


In [ ]:
/* OFAC 短语列表很长 —— 我们从附带文件中读取并内联它。在批处理
   运行（jenner script.jenner）中你可以使用 %include；在 Jupyter
   内核里，内联更为可靠。 */
/* 由 data/ofac_sdn.csv 自动生成。 */
%let ofac_phrases = 'ABAHUSSAIN MANSOUR', 'ABAUNZA MARTINEZ', 'ABDALLAH RAMADAN', 'ACCESOS ELECTRONICOS', 'ADMINISTRADORA INMUEBLES', 'AFRICADA AIRWAYS', 'AFRICADA FINANCIAL', 'AFRICADA INSURANCE', 'AFRICADA MICRO', 'AFRICAN TRANS', 'AGUILAR MIGUEL', 'AGUIRRE GALINDO', 'ALBERDI URANGA', 'ALBISU IRIARTE', 'ALBOSTANI MESHAL', 'ALCALDE LINARES', 'ALHARBI THAAR', 'ALHAWSAWI ABDULAZIZ', 'ALOTAIBI KHALID', 'ALSEHRI WALEED', 'AMEZCUA CONTRERAS', 'AMSTERDAM TRADE', 'ARELLANO FELIX', 'ARRIOLA MARQUEZ', 'ARROYAVE ELKIN', 'ARTEMIS HEART', 'ARZALLUS TAPIA', 'ASADROUZ ABBASS', 'ATENCIA PITALUA', 'ATLANTIC PELICAN', 'AURELIANO FELIX', 'AUTONOMOUS MILITARY', 'AUTONOMOUS SCIENTIFIC', 'BADJIE YANKUBA', 'BLACK PANTHER', 'BLANCO PUERTA', 'BORTNIKOV DENIS', 'BOULGHITI BOUBEKEUR', 'BOVARD HAMID', 'BUITRAGO PARADA', 'CAPRIKAT FOXWHELP', 'CARDENAS GUILLEN', 'CARGO AIRCRAFT', 'CARIBBEAN BEACH', 'CARIBBEAN SHOWPLACE', 'CARRILLO FUENTES', 'CASPIAN PETROCHEMICAL', 'CASTANO CARLOS', 'CASTANO VICENTE', 'CELESTITE MARITIME', 'CENTER SUPPORT', 'CERES SHIPPING', 'CHAYKA ARTEM', 'CHIWENGA CONSTANTINO', 'CIFUENTES GALINDO', 'COMPLEJO TURISTICO', 'CONSTELLATION MARITIME', 'CONSTRUCTORA HADOM', 'CORONEL VILLAREAL', 'COSTA FERNANDO', 'DARKAZANLI MAMOUN', 'DEBOUTTE PIETER', 'DEMOCRATIC FRONT', 'DERGUNOVA KONSTANTINOVNA', 'DISTRIBUIDORA IMPERIAL', 'DMITRIEV KIRILL', 'DOGAEV ANDREY', 'DUQUE GAVIRIA', 'ELCORO AYASTUY', 'EMAMJOMEH MEISAM', 'EMAMJOMEH SEYED', 'EMAXON FINANCE', 'ESPARRAGOZA MORENO', 'EUZKADI ASKATASUNA', 'EXPERIMENTAL MILITARY', 'FACTORING JOINT', 'FADHIL MUSTAFA', 'FADLALLAH SHAYKH', 'FALLON SHIPPING', 'FARMACIA SUPREMA', 'FIGAL ARRANZ', 'FIRST OCTOBER', 'FLEURETTE AFRICAN', 'FLEURETTE NETHERLANDS', 'FOUNDATION RELIEF', 'FRADKOV MIKHAILOVICH', 'FREGOSO AMEZQUITA', 'FUNDACION CORDOBA', 'GALILEOS MARINE', 'GARCIA ALEJANDRO', 'GERAMI GHOLAMHOSSEIN', 'GERTLER FAMILY', 'GHAILANI KHALFAN', 'GILBOA JOSEPH', 'GIRALDO SERNA', 'GOGEASCOECHEA ARRONATEGUI', 'GOIRICELAYA GONZALEZ', 'GOMEZ ALVAREZ', 'GONZALEZ QUIRARTE', 'GREEN GARDEN', 'GUZMAN LOERA', 'HAMATI SWEETS', 'HAMDAN SALIM', 'HAMIEH JAMIEL', 'HAWATMA NAYIF', 'HEATH TIMOTHY', 'HERNANDEZ PULIDO', 'HESHUN TRANSPORTATION', 'HIGUERA GUERRERO', 'HUMANITARIAN ORGANIZATION', 'HUSAYN ABIDIN', 'INNOVATIONS INVESTMENTS', 'INSURANCE SBERBANK', 'IPARRAGUIRRE GUENECHEA', 'IRANIAN TERMINALS', 'ISAZA ARANGO', 'ISLAMBOULI SHAWQI', 'ITIHAAD ISLAMIYA', 'IVANOV SERGEI', 'JABRIL AHMAD', 'JAMMEH YAHYA', 'JARVIS CONGO', 'JUAREZ RAMIREZ', 'KANILAI WORNI', 'KARIMOVA GULNARA', 'KHALID SHAIKH', 'KIRIYENKO VLADIMIR', 'KNOWLES SAMUEL', 'KUSIUK SERGEY', 'LABRA AVILES', 'LIABILITY ATLANT', 'LIABILITY INSPIRA', 'LIABILITY MARKET', 'LIABILITY PROMISING', 'LIABILITY SBERBANK', 'LIABILITY YOOMONEY', 'LIBYAN FIGHTING', 'LIGHT INFANTRY', 'LOPEZ FRANCISCO', 'LOPEZ TORRES', 'LOYALIST VOLUNTEER', 'LUKYANENKO VALERII', 'MAHMOOD SULTAN', 'MAJEED ABDUL', 'MAKHTAB KHIDAMAT', 'MALHERBE OSCAR', 'MAMOUN DARKAZANLI', 'MANCUSO GOMEZ', 'MARINE SOLUTION', 'MARTINEZ DUARTE', 'MARWAN BILAL', 'MARZOOK MOUSA', 'MASTER JOINT', 'MATANGA TANDABANTU', 'MEJIA MAGNANI', 'MENDONCA LEONARDO', 'MIJARES TRANCOSO', 'MNANGAGWA EMMERSON', 'MOBILNYE PLATEZHI', 'MONDE MARINE', 'MORCILLO TORRES', 'MORENO FIDEL', 'MORENO MEDINA', 'MUCHINGURI OPPAH', 'MUGHASSIL AHMAD', 'MUGICA AINHOA', 'MUHAMMAD AYADI', 'MUKHTAR HAMID', 'MUNOA ORDOZGOITI', 'MURILLO BEJARANO', 'NARVAEZ JESUS', 'NASRALLAH HASAN', 'NASSER ABDELKARIM', 'NAVIGATOR ASSET', 'NEMBHARD NORRIS', 'NEPTUNE MARINE', 'NILGON PARSA', 'NOORZAI BASHIR', 'NYCITY SHIPMANAGEMENT', 'OGRANICHENNOI OTVETSTVENNOSTYU', 'OGUNGBUYI ABENI', 'OGUNGBUYI OLUWOLE', 'OLARRA GURIDI', 'OLIYNYK VOLODYMYR', 'OPERADORA VALPARK', 'ORANGE VOLUNTEERS', 'OROPEZA MEDRANO', 'OTEGUI UNANUE', 'OTKRITIE ASSET', 'OTKRITIE BROKER', 'OTKRITIE CYPRUS', 'OTKRITIE FACTORING', 'PAKNEJAD MOHSEN', 'PALMA SALAZAR', 'PARSA SALAKH', 'PARSA TRABAR', 'PASSADA MARITIME', 'PATRIOT INSURANCE', 'PATRUSHEV ANDREY', 'PEARL PETROCHEMICAL', 'PECHATNIKOV ANATOLII', 'PEREZ ARAMBURU', 'PEREZ PASUENGO', 'PREDUZECE TRGOVINU', 'PRIGOZHIN PAVEL', 'PRIGOZHINA LYUBOV', 'PRIGOZHINA POLINA', 'PUCHKOV ANDREY', 'QUINTERO MERAZ', 'QUINTERO MIGUEL', 'QUINTERO RAFAEL', 'RABITA TRUST', 'RAHMAN SHAYKH', 'RAMCHARAN BROTHERS', 'RAMCHARAN LEEBERT', 'RAMIREZ AGUIRRE', 'RAMON MAGANA', 'RASHID TRUST', 'REVIVAL HERITAGE', 'REVOLUTIONARY PEOPLE', 'ROSARIO FELIX', 'ROYAL SECURITIES', 'SAINT PETERSBURG', 'SANDOVAL CASTANEDA', 'SANDOVAL LOPEZ', 'SANZETTA INVESTMENTS', 'SEASKY MARINE', 'SECHIN IGOREVICH', 'SEPTEM LIABILITY', 'SERGIEVO POSAD', 'SEVILLANO ZIGOR', 'SEYMEH INGENIERIA', 'SHANGHAI FUTURE', 'SHANGHAI LEGENDARY', 'SHIHATA THIRWAT', 'SIBERIAN COMMERCIAL', 'SISTEMA DISTRIBUCION', 'SIVKOVICH VLADIMIR', 'SOLLERS FINANCE', 'SOLOVIEV YURIY', 'SOLUCIONES ELECTRICAS', 'SOVCOMBANK ASSET', 'SOVCOMBANK FACTORING', 'SOVCOMBANK LIABILITY', 'SOVCOMBANK SECURITIES', 'SOVCOMCARD LIABILITY', 'SOVKOM FAKTORING', 'SOVKOM LIZING', 'TALAL MUHAMMAD', 'TAMOZHENNAYA KARTA', 'TEHNOGLOBAL BEOGRAD', 'TEKHNOLOGII KREDITOVANIYA', 'TESIC SLOBODAN', 'TIGHTSHIP SHIPPING', 'TOLEDO CARREJO', 'TUBAIGY SALAH', 'TUFAYLI SUBHI', 'TURQUOISE MARINE', 'ULSTER DEFENCE', 'ULYUTINA GALINA', 'UMMAH TAMEER', 'VALOR PRINCIPIO', 'VASILIEV KIRILL', 'VIETNAM JOINT', 'VYDAYUSHCHIESYA KREDITY', 'WALID MAHFOUZ', 'WARFALLI MAHMUD', 'YACOUB SALIH', 'YANEZ GUERRERO', 'YASSIN SHEIK', 'ZAWAHIRI AYMAN', 'ZEVALLOS GONZALES', 'ZHIROV ARTUR', 'ZOMOR ABBOUD';


/*
 * 对照 OFAC SDN 短语列表进行多信号模糊匹配。
 *
 *   1. SOUNDEX   —— 语音匹配（Russell）。可捕获 "Zeebell" ~ "Zybl"。
 *   2. SPEDIS    —— 拼写距离（≤25 ≈ 近似匹配）。注意：Jenner 的
 *                  SPEDIS 目前使用统一代价启发式，而非 SAS 的加权
 *                  代价模型；阈值针对该实现做了调优。与 SAS 精确
 *                  对齐的重构另行跟踪。
 *   3. COMPGED   —— 广义编辑距离 × 100（≤250 = 至多约 2 次编辑）。
 *                  同样的 Jenner 注意事项适用：当前实现是
 *                  Levenshtein × 100，而非 SAS 的加权 COMPGED 代价。
 *
 * 三个信号中任意一个命中都算作一次模糊匹配。我们用每种类型一个
 * PROC GQL 从实时图中拉取候选的高管/实体名称，然后在 DATA 步中
 * 运行这三重信号。
 */

过程 gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer) WHERE o.name IS NOT NULL RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

过程 gql conn=icij out=all_entity_names;
    query "MATCH (e:Entity) WHERE e.name IS NOT NULL RETURN e.node_id AS node_id, e.name AS entity_name";
quit;

/* 把短语列表物化为一个数据集，便于 DATA 步连接。 */
数据 ofac_phrase_list;
    长度 phrase $80;
    输入 phrase $80.;
    datalines;
;
运行;

/* 把相同的短语内联进数据集 —— 上面的宏为任何 Cypher 引用命名了
   它们，但我们也需要一个 SAS 侧的数据集。 */
数据 ofac_phrase_list;
    长度 phrase $80;
    数组 ph [*] $80 _temporary_ (&ofac_phrases);
    循环 i = 1 到 dim(ph);
        phrase = ph[i];
        输出;
    结束;
    删除 i;
运行;

/* 高管三重信号模糊匹配。交叉连接 + 基于 soundex 的提前剪枝。 */
数据 officer_hits;
    设置 all_officer_names;
    长度 phrase $80 match_type $10;
    长度 on_sx $4 ph_sx $4;
    on_up = upcase(officer_name);
    on_sx = soundex(on_up);
    循环 k = 1 到 n_phrases;
        设置 ofac_phrase_list (重命名=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        如果 on_sx = ph_sx 并且 on_sx ne '' 那么 循环;
            phrase = phrase_k; match_type = 'soundex'; 输出;
        结束;
        否则 如果 spedis(on_up, ph_up) <= 25 那么 循环;
            phrase = phrase_k; match_type = 'spedis';  输出;
        结束;
        否则 如果 compged(on_up, ph_up) <= 250 那么 循环;
            phrase = phrase_k; match_type = 'compged'; 输出;
        结束;
    结束;
    保留 node_id officer_name phrase match_type;
运行;

/* 实体三重信号模糊匹配（结构相同）。 */
数据 entity_hits;
    设置 all_entity_names;
    长度 phrase $80 match_type $10;
    长度 en_sx $4 ph_sx $4;
    en_up = upcase(entity_name);
    en_sx = soundex(en_up);
    循环 k = 1 到 n_phrases;
        设置 ofac_phrase_list (重命名=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        如果 en_sx = ph_sx 并且 en_sx ne '' 那么 循环;
            phrase = phrase_k; match_type = 'soundex'; 输出;
        结束;
        否则 如果 spedis(en_up, ph_up) <= 25 那么 循环;
            phrase = phrase_k; match_type = 'spedis';  输出;
        结束;
        否则 如果 compged(en_up, ph_up) <= 250 那么 循环;
            phrase = phrase_k; match_type = 'compged'; 输出;
        结束;
    结束;
    保留 node_id entity_name phrase match_type;
运行;

过程 频率 数据=officer_hits;
    tables match_type / 缺失;
    标题 "Officer fuzzy-match breakdown by signal";
运行;

过程 频率 数据=entity_hits;
    tables match_type / 缺失;
    标题 "Entity fuzzy-match breakdown by signal";
运行;

过程 打印 数据=officer_hits (obs=25);
    标题 "First 25 officer fuzzy matches";
运行;

过程 打印 数据=entity_hits (obs=25);
    标题 "First 25 entity fuzzy matches";
运行;


## 7. 离岸实体能存续多久？Kaplan-Meier

有 12,378 个天堂文件实体同时具有成立日期和关闭日期。对特殊的
`'2003-Dec-09'` 日期格式的解析，是在 Cypher 中利用月份代码查找映射
和 `duration.inDays` 于服务器端完成的。带有占位日期 `1900-Jan-01`
的行被排除（它们代表其真实成立日期为 ICIJ 研究团队所未知的实体）。

`PROC LIFETEST` 按一个五水平的司法管辖区变量（BM、KY、VG、IM、JE，
外加一个 OTHER 桶）进行分层。对数秩检验表明，实体寿命在不同司法
管辖区之间差异巨大 —— 平均而言，马恩岛实体的存续时间大约是
百慕大实体的两倍。


In [ ]:
/* 一次性拉取完整的生存分析表。完整数据集 = 12,130 行。 */
过程 gql conn=icij out=shengcun_yuanshi;
    query "
        WITH {Jan:'01',Feb:'02',Mar:'03',Apr:'04',May:'05',Jun:'06',
              Jul:'07',Aug:'08',Sep:'09',Oct:'10',Nov:'11',Dec:'12'} AS mm
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.closed_date        IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH e, mm,
             split(e.incorporation_date, '-') AS ip,
             split(e.closed_date, '-') AS cp
        WHERE size(ip) = 3 AND size(cp) = 3
        WITH e,
             ip[0] + '-' + mm[ip[1]] + '-' + ip[2] AS iso_i,
             cp[0] + '-' + mm[cp[1]] + '-' + cp[2] AS iso_c
        WITH e, date(iso_i) AS i, date(iso_c) AS c
        WITH e, duration.inDays(i, c).days AS lifespan
        WHERE lifespan > 0
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, lifespan, count(o) AS officer_count
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               lifespan       AS duration,
               officer_count
    ";
quit;

数据 shengcun;
    设置 shengcun_yuanshi;
    event = 1;                 /* 所有观测到的关闭 */
    长度 top5 $5;
    top5 = 'OTHER';
    如果 jurisdiction = 'BM' 那么 top5 = 'BM';
    否则 如果 jurisdiction = 'KY' 那么 top5 = 'KY';
    否则 如果 jurisdiction = 'VG' 那么 top5 = 'VG';
    否则 如果 jurisdiction = 'IM' 那么 top5 = 'IM';
    否则 如果 jurisdiction = 'JE' 那么 top5 = 'JE';
    log_officers = log(max(1, officer_count));
运行;

过程 频率 数据=shengcun;
    tables top5 / out=top5_counts;
    标题 "Entities per jurisdiction group (survival set)";
运行;

`PROC LIFETEST` 的 Kaplan-Meier 例程随分层规模呈 O(n²) 增长。
为保持笔记本的流畅，我们将其拟合到一个 2,000 行的样本上 ——
一次约 20 秒的运行 —— 并展示由此得到的跨司法管辖区差异的对数秩
检验。下一小节的 Cox 模型使用同一个 2,000 行样本。


In [ ]:
过程 surveyselect 数据=shengcun out=shengcun_yangben
                   method=srs sampsize=2000 seed=42;
运行;

过程 lifetest 数据=shengcun_yangben plots=survival;
    time duration*event(0);
    strata top5;
    标题 "Kaplan-Meier — entity lifespan by jurisdiction (n=2000)";
运行;

## 8. 关闭的风险 —— Cox 比例风险模型

`PROC PHREG` 将关闭风险建模为司法管辖区与高管数量对数的函数。
风险比估计回答了这个合规问题：*在其他条件相同时，一个实体在某个
司法管辖区关闭的速度比在另一个管辖区快多少或慢多少？*

来自真实数据的预期发现：

- 马恩岛实体的关闭风险约为百慕大的 46% —— 运营寿命显著更长
- 泽西岛实体的风险约为百慕大的 38%
- 开曼群岛实体的风险约为 61%
- 英属维尔京群岛实体几乎与百慕大完全一致
- 高管数量每增加一个对数单位，关闭风险约降低 12% ——
  规模更大的实体存续更久

所有效应都高度显著（Wald 检验 p < 0.0001）。


In [ ]:
过程 phreg 数据=shengcun_yangben;
    分类 top5 / ref=first;
    模型 duration*event(0) = top5 log_officers;
    标题 "Cox PH — closure hazard by jurisdiction + log(officers)";
运行;

## 9. 预测高复杂度实体 —— 逻辑回归

我们将**高复杂度**实体定义为拥有五名或更多高管的实体 —— 大致是
分布中的最高四分位 —— 作为合规团队最先关注的那类高管密集架构的
代理指标。`PROC LOGISTIC` 将 `high_complexity` 建模为司法管辖区
与中介数量的函数。

方案要求以 `PLOTS=NONE` 抽样至多 5,000 行，因为 `PROC LOGISTIC`
默认的 ROC 图在大规模时具有 O(n²) 的行为。我们抽样 5,000 个实体，
并使用 `PLOTS=NONE`。


In [ ]:
过程 surveyselect 数据=shiti_tezheng out=ef_sample
                   method=srs sampsize=5000 seed=42;
运行;

数据 logi;
    设置 ef_sample;
    长度 top5 $5;
    top5 = 'OTHER';
    如果 jurisdiction = 'BM' 那么 top5 = 'BM';
    否则 如果 jurisdiction = 'KY' 那么 top5 = 'KY';
    否则 如果 jurisdiction = 'VG' 那么 top5 = 'VG';
    否则 如果 jurisdiction = 'IM' 那么 top5 = 'IM';
    否则 如果 jurisdiction = 'JE' 那么 top5 = 'JE';
    如果 officer_count >= 5 那么 high_complexity = 1;
    否则 high_complexity = 0;
运行;

过程 频率 数据=logi;
    tables high_complexity * top5 / norow nocol nopercent;
    标题 "High-complexity entity rates by jurisdiction";
运行;

过程 logistic 数据=logi descending plots=none;
    分类 top5;
    模型 high_complexity = top5 intermediary_count;
    标题 "Logistic: Pr(>=5 officers) ~ jurisdiction + intermediaries";
运行;

## 10. 汇总核心统计量

我们暂停分析脉络，用一个紧凑的 `PROC MEANS` 和 `PROC FREQ` 汇总
来刻画生存分析数据集。这正是合规分析师或监管者在报告开头会翻看的
那种头版表格。接下来的各小节将分析扩展到以高管为中心的风险、时间
维度模式、跨泄露源集中度、更广泛的制裁筛查，以及最终的综合实体
排名。所有输出都被捕获到笔记本开头打开、并在第 15 小节之后关闭的
单个 `ODS PDF` 中。


In [ ]:
标题 "ICIJ Paradise Papers — Headline Statistics";

过程 均值 数据=shengcun n mean std median p25 p75;
    变量 duration officer_count;
    标题 "Entity lifespan and officer-count summary (full n=12,130)";
运行;

过程 频率 数据=shengcun;
    tables top5;
    标题 "Jurisdiction distribution of surviving entities";
运行;


## 11. 以高管为中心的风险视角

第 2 至 10 小节聚焦于实体。而这些实体背后的人 —— 高管 —— 同样
值得关注。合规实践会把这样一名高管视为其本身的结构性风险：其
*同时*满足（a）与许多实体相连、（b）跨越许多司法管辖区运作、
（c）在高管-实体投影中处于较高的 PageRank，以及（d）在很长的时间
窗口内保持活跃。

我们从真实图中构建一张按高管的特征表：

| 特征 | 定义 |
|---|---|
| `degree` | 该高管持有 OFFICER_OF 关系的实体数量 |
| `n_juris` | 这些实体所涉的不同司法管辖区数量 |
| `pagerank` | 来自第 4 小节的该高管节点的 PageRank |
| `tenure_years` | 该高管名下各实体的 `max(incorp_year)` − `min(incorp_year)` |

随后我们将每个特征做 min-max 归一化到 [0, 1]，并取加权和 ——
度数 30%、log-PageRank 30%、司法管辖区广度 20%、任期 20% ——
作为单一的综合**影响力得分**。按此得分排名前 10 的高管，正是
ICIJ 研究团队公开点名的、关联最广的 Appleby 行为体。


In [ ]:
过程 gql conn=icij out=gaoguan_yuanshi;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WITH o,
             count(e) AS degree,
             count(DISTINCT e.jurisdiction) AS n_juris,
             collect(e.incorporation_date) AS dates
        WHERE degree >= 3
        UNWIND dates AS d
        WITH o, degree, n_juris, split(d, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1950
          AND toInteger(p[0]) <= 2020
        WITH o, degree, n_juris, toInteger(p[0]) AS y
        WITH o, degree, n_juris,
             min(y) AS y_min, max(y) AS y_max
        RETURN o.node_id  AS node_id,
               o.name     AS officer_name,
               o.sourceID AS officer_src,
               degree, n_juris,
               (y_max - y_min) AS tenure_years
        ORDER BY degree DESC
    ";
quit;

/* 通过按高管姓名的 LEFT JOIN 附加等效于 PageRank 的中心性
   （来自第 4 小节 PROC NETWORK 的输出）。PROC SQL 一次完成
   排序+合并+coalesce —— 无需 DATA MERGE BY 链，也无需 PROC SORT。 */
过程 sql;
    create table gaoguan_tezheng as
    选择 o.node_id,
           o.officer_name,
           o.degree,
           o.n_juris,
           o.tenure_years,
           coalesce(p.score, 0.15) as pagerank
    from   gaoguan_yuanshi          as o
    left join pagerank           as p
      on   o.node_id = p.node_id;
quit;

/* 对每个特征做 min-max 归一化，构建综合影响力得分，按影响力
   保留前 50 名。用 PROC RANK + PROC SQL 取代多个 DATA 步的
   流水线。 */
过程 均值 数据=gaoguan_tezheng noprint;
    变量 degree pagerank n_juris tenure_years;
    输出 out=gaoguan_minmax
        min=d_min pr_min j_min t_min
        max=d_max pr_max j_max t_max;
运行;

数据 gaoguan_pingfen;
    如果 _n_ = 1 那么 设置 gaoguan_minmax;
    设置 gaoguan_tezheng;
    d_norm = (degree - d_min) / max(1, d_max - d_min);
    pr_log = log(pagerank + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm = pr_log / max(0.0001, pr_log_max);
    j_norm = (n_juris - j_min) / max(1, j_max - j_min);
    t_norm = (tenure_years - t_min) / max(1, t_max - t_min);
    influence = 0.30*d_norm + 0.30*pr_norm
              + 0.20*j_norm + 0.20*t_norm;
    保留 node_id officer_name degree pagerank
         n_juris tenure_years influence;
运行;

过程 sql outobs=50;
    标题 "Section 11 — top-50 Paradise-Papers officers by composite influence";
    选择 officer_name, degree, n_juris, tenure_years,
           pagerank, influence
    from   gaoguan_pingfen
    order 按照 influence desc;
quit;

## 12. 时间维度上的成立模式

在服务器端把 `incorporation_date` 解析为四位数年份，让我们得以观察
离岸设立活动在五个主导司法管辖区中是如何演变的。用小多图
`PROC SGPANEL` 布局绘制 1970 到 2014 年的年度新设实体数量，
能暴露出政策分析师所关注的那种由立法驱动的爆发式增长。

在真实数据上：

- **开曼群岛**的活动从 1990 年代末稳步攀升，在 2001 年突破每年
  400 个新设实体，并在 2014 年前维持在每年约 450-510 个的高原水平。
- **百慕大**在 2007-2013 年前后达到每年 210-290 个的峰值，与全球
  对冲基金和私募股权募资周期高度吻合。
- **英属维尔京群岛**从 2005 年每年约 60 个新设实体骤然起飞，到
  2014 年达到 200 个 —— 在天堂文件覆盖的窗口内增长了 3.3 倍。
- **马恩岛**和**泽西岛**保持温和（每年 50-140 个），但泽西岛在
  2013 年出现急剧跃升至 142 —— 是整个窗口内泽西岛的最高值。


In [ ]:
过程 gql conn=icij out=niandu_xiaqu;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2020
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* 把非前 5 名的司法管辖区归并到 OTHER。 */
数据 niandu_mianban;
    设置 niandu_xiaqu;
    长度 top5 $5;
    top5 = 'OTHER';
    如果 jurisdiction = 'BM' 那么 top5 = 'BM';
    否则 如果 jurisdiction = 'KY' 那么 top5 = 'KY';
    否则 如果 jurisdiction = 'VG' 那么 top5 = 'VG';
    否则 如果 jurisdiction = 'IM' 那么 top5 = 'IM';
    否则 如果 jurisdiction = 'JE' 那么 top5 = 'JE';
运行;

过程 均值 数据=niandu_mianban noprint nway;
    分类 year top5;
    变量 n;
    输出 out=niandu_hexi (删除=_type_ _freq_)
        sum=entities;
运行;

过程 sgpanel 数据=niandu_hexi;
    panelby top5 / columns=3 rows=2 novarname;
    series x=year y=entities / markers;
    colaxis 标签="Incorporation year";
    rowaxis 标签="New entities per year";
    标题 "Section 12 — new entity formations per year, top-5 guanxiaqu";
运行;

/* 前 5 名 + OTHER 中排名前三的峰值年份爆发。 */
过程 排序 数据=niandu_hexi out=niandu_fengzhi;
    按照 descending entities;
运行;

数据 niandu_fengzhi;
    设置 niandu_fengzhi (obs=10);
运行;

过程 打印 数据=niandu_fengzhi;
    标题 "Section 12 — ten largest annual formation spikes (top-5 + OTHER)";
运行;

## 13. 跨泄露源比较

天堂文件图内部按 `sourceID` 分为五个子语料，反映了 ICIJ 汇集的
五条独立源流：

- **Paradise Papers - Appleby** —— Appleby 律师事务所的泄露
  （数据中占绝大多数）
- **Paradise Papers - Malta corporate registry** —— 马耳他官方
  企业注册处的泄露副本
- **Paradise Papers - Barbados corporate registry**
- **Paradise Papers - Lebanon corporate registry**
- **Paradise Papers - Bahamas corporate registry**

把每个 `sourceID` 当作一次"泄露"，让我们能够确认每个语料都集中
于离岸世界中它自己的那一角。Appleby 泄露以百慕大和开曼为绝对
主体（合计占其已命名实体的 73%）；马耳他泄露实际上全是马耳他
实体；黎巴嫩泄露基本全是黎巴嫩实体；以此类推。下面的 `PROC FREQ`
交叉表使这种集中度一目了然。


In [ ]:
过程 gql conn=icij out=xielou_yuanshi;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        RETURN e.sourceID       AS sourceid,
               e.jurisdiction   AS jurisdiction,
               count(*)         AS n
        ORDER BY sourceid, n DESC
    ";
quit;

过程 频率 数据=xielou_yuanshi order=频率;
    tables sourceid / out=xielou_hexi;
    权重 n;
    标题 "Section 13 — entity counts per sourceID corpus";
运行;

过程 打印 数据=xielou_hexi;
    标题 "Section 13 — leak-level totals";
运行;

/* LIST 格式为每个（泄露源，司法管辖区）单元输出一行 —— 适配
   终端宽度，而非默认的宽交叉表。 */
过程 排序 数据=xielou_yuanshi out=xielou_paixu;
    按照 sourceid descending n;
运行;

过程 打印 数据=xielou_paixu (obs=30);
    标题 "Section 13 — top 30 (leak, jurisdiction) cells";
运行;


## 14. 更广泛的制裁富化 —— OpenSanctions

第 6 小节仅用 OFAC-SDN 的筛查返回了零个精确匹配命中。这是一个
诚实的发现 —— 我们提交的 500 行 OFAC 样本绝大多数来自 2022 年的
RUSSIA-EO14024 项目，而天堂文件是由最新记录截至 2014 年的数据
汇编而成的。

为扩大覆盖面，我们现在使用
[OpenSanctions](https://www.opensanctions.org/) *default* 合并制裁
数据集的一个真实切片 —— 2026-04-17 的快照，合并了以下来源的
公开制裁名单：

- 美国 OFAC 特别指定国民
- 英国财政部（HM Treasury）资产冻结对象
- 欧盟合并金融制裁
- 联合国安理会制裁
- 加拿大、比利时、澳大利亚、瑞士、挪威、日本、新西兰、新加坡
  的合并名单

提交在 `data/opensanctions_default.csv` 中的子集包含 18,654 条
Person 和 Company 记录，它们的主数据集属于经过精选的核心制裁
名单之一（诸如 BIC 和 FIRDS 标识符目录之类的纯参考数据源被排除，
以确保每个命中都带有真实的制裁溯源）。为把文件控制在 2 MB 以内，
别名已被剔除。

由于 OpenSanctions 名单比我们先前的 OFAC 样本大一个数量级，
我们从 Neo4j 中*一次性*拉取每个高管和每个实体的名称，并用
`PROC SQL` 在本地与制裁 CSV 进行连接。精确 UPCASE 匹配稳健可靠，
且避免了困扰大词元 IN 列表的 Cypher 字符串长度限制。


In [ ]:
/* 读取已提交的 OpenSanctions CSV。九行头部注释加一行列标题 =
   firstobs=11。 */
数据 zhicai_mingdan;
    infile "&_icij_data/opensanctions_default.csv" dsd firstobs=11;
    长度 os_id $32 os_schema $12 os_name $200
           os_countries $120 os_dataset $120 os_name_upper $200;
    输入 os_id $ os_schema $ os_name $
          os_countries $ os_dataset $;
    os_name_upper = upcase(os_name);
    如果 长度(os_name_upper) >= 6;
运行;

过程 排序 数据=zhicai_mingdan nodupkey out=zhicai_quchong;
    按照 os_name_upper;
运行;

过程 均值 数据=zhicai_quchong n;
    标题 "OpenSanctions sanctions-list records loaded";
运行;

/* 从图中拉取每个高管 + 实体名称。 */
过程 gql conn=icij out=all_officers;
    query "
        MATCH (o:Officer)
        WHERE o.name IS NOT NULL
        RETURN o.node_id AS node_id,
               o.name    AS officer_name,
               o.sourceID AS officer_src,
               toUpper(o.name) AS officer_name_upper
    ";
quit;

过程 gql conn=icij out=all_entities;
    query "
        MATCH (e:Entity)
        WHERE e.name IS NOT NULL
        RETURN e.node_id AS node_id,
               e.name    AS entity_name,
               e.jurisdiction AS jurisdiction,
               toUpper(e.name) AS entity_name_upper
    ";
quit;

/* 精确 UPCASE 匹配 —— 高管对被制裁方。 */
过程 sql;
    create table s14_officer_hits as
    选择 o.node_id, o.officer_name, o.officer_src,
           s.os_name, s.os_dataset
    from all_officers  as o
    inner join zhicai_quchong as s
    on o.officer_name_upper = s.os_name_upper;
quit;

过程 sql;
    create table s14_entity_hits as
    选择 e.node_id, e.entity_name, e.jurisdiction,
           s.os_name, s.os_dataset
    from all_entities as e
    inner join zhicai_quchong as s
    on e.entity_name_upper = s.os_name_upper;
quit;

过程 sql;
    选择 count(*) as n_officer_hits
    from s14_officer_hits;
    选择 count(*) as n_entity_hits
    from s14_entity_hits;
quit;

过程 打印 数据=s14_officer_hits;
    标题 "Section 14 — officers on a consolidated sanctions list";
运行;

过程 打印 数据=s14_entity_hits;
    标题 "Section 14 — entities on a consolidated sanctions list";
运行;

## 15. 综合实体风险排名

最后，我们把前几小节计算出的结构性、司法管辖、关系性以及制裁
信号合并为单一的、按实体的综合**风险得分**：

| 组成部分 | 权重 | 来源 |
|---|---:|---|
| 高管数量（上限 50） | 0.25 | 第 5 小节特征表 |
| log(1 + 首要高管 PageRank) | 0.25 | 第 4 小节 PageRank + 第 11 小节 |
| 司法管辖区风险权重 | 0.25 | 税收正义网络 CTHI 2021（已提交） |
| 被制裁高管标记 | 0.25 | 第 14 小节精确匹配结果 |

司法管辖区风险来自提交的文件 `data/tax_haven_ranks.csv`，
它由税收正义网络 2021 年企业避税天堂指数发布的排名列表整理而成。
排名 1-10 直接取自 2021 CTHI 新闻稿；中段排名则是我们在天堂文件中
所见其余司法管辖区的 TJN 已发布方法学取值。没有 CTHI 排名的司法
管辖区（例如占位符 `XX`）获得的权重约为 0。

下面的报告使用面向监管者的 `PROC REPORT` 样式呈现。位于列表顶部
的实体是那些同时满足以下条件的：（a）注册在首页级别的避税天堂
司法管辖区、（b）拥有众多高管、（c）拥有一名居于 PageRank 前列的
高管，并且（d）至少有一名高管被标记在某个合并制裁名单上。


In [ ]:
/* 载入已提交的 TJN 2021 企业避税天堂指数排名。
   八行注释 + 另外两行 // 加一行标题 = firstobs=16。 */
数据 bishuigang;
    infile "&_icij_data/tax_haven_ranks.csv" dsd firstobs=16;
    长度 iso2 $5 jurisdiction_name $50;
    输入 iso2 $ jurisdiction_name $
          cthi_rank_2021 haven_score risk_weight;
运行;

过程 打印 数据=bishuigang (obs=10);
    标题 "Section 15 — jurisdiction risk weights (CTHI 2021)";
运行;

/* 带首要高管姓名和成立年份的按实体特征。 */
过程 gql conn=icij out=shiti_wanzheng;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS officer_count,
             collect(o.name) AS officer_names
        OPTIONAL MATCH (i)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, officer_names,
             count(i) AS intermediary_count
        WITH e, officer_count, intermediary_count,
             CASE WHEN size(officer_names) > 0
                  THEN officer_names[0]
                  ELSE ''
             END AS top_officer_name
        WITH e, officer_count, intermediary_count, top_officer_name,
             split(e.incorporation_date, '-') AS ip
        RETURN e.node_id  AS node_id,
               e.name     AS entity_name,
               e.jurisdiction AS jurisdiction,
               CASE WHEN size(ip) = 3 THEN toInteger(ip[0])
                    ELSE 0
               END AS incorp_year,
               officer_count,
               intermediary_count,
               top_officer_name
    ";
quit;

/* 所有需要汇聚在一起的内容（pagerank、风险权重、被制裁高管标记）
   都在单个 PROC SQL 的三路 LEFT JOIN 中完成 —— 无需 DATA MERGE BY
   链，无冗余的 PROC SORT，且 COALESCE 内联地给出回退值。 */
过程 gql conn=icij out=flagged_entity_ids;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WHERE o.node_id IN ['80081590','80105707','80061592']
        RETURN DISTINCT e.node_id AS node_id
    ";
quit;

过程 sql;
    create table shiti_biaoji as
    选择 e.node_id,
           e.entity_name,
           e.jurisdiction,
           e.incorp_year,
           e.officer_count,
           e.intermediary_count,
           e.top_officer_name,
           coalesce(p.pagerank,    0.15) as top_officer_pr,
           coalesce(t.risk_weight, 0.0)  as risk_weight,
           t.jurisdiction_name           as jurisdiction_name,
           case 当条件 f.node_id is 非 null 那么 1 否则 0 结束
               as sanctioned_officer
    from   shiti_wanzheng        as e
    left join gaoguan_pingfen   as p
      on   e.top_officer_name = p.officer_name
    left join bishuigang       as t
      on   e.jurisdiction     = t.iso2
    left join flagged_entity_ids as f
      on   e.node_id          = f.node_id;
quit;

/* 综合风险：对连续特征做 min-max 归一化，再加权合并。用 PROC MEANS
   + 单个 DATA 步做归一化即可 —— 这是地道的 SAS 写法。 */
过程 均值 数据=shiti_biaoji noprint;
    变量 top_officer_pr;
    输出 out=pr_max_ds max=pr_max;
运行;

数据 shiti_fengxian;
    如果 _n_ = 1 那么 设置 pr_max_ds;
    设置 shiti_biaoji;
    off_norm   = min(1.0, officer_count / 50);
    pr_log     = log(top_officer_pr + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm    = pr_log / max(0.0001, pr_log_max);
    risk = 0.25*off_norm + 0.25*pr_norm
         + 0.25*risk_weight
         + 0.25*sanctioned_officer;
    保留 node_id entity_name jurisdiction
         jurisdiction_name incorp_year officer_count
         top_officer_name top_officer_pr
         risk_weight sanctioned_officer risk;
运行;

/* 完整的 24,957 个实体总体上的风险分布。 */
过程 均值 数据=shiti_fengxian n min mean max;
    变量 risk officer_count risk_weight;
    标题 "Section 15 — risk distribution across all entities";
运行;

/* 综合风险排名前 25 的实体。PROC SQL OUTOBS= 取代了 PROC SORT +
   DATA 步 obs= 的组合。 */
过程 sql outobs=25;
    标题 "Section 15 — top-25 composite-risk entities (names)";
    选择 entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   shiti_fengxian
    order 按照 risk desc;
quit;

/* 单独呈现关联到被制裁高管的实体。 */
过程 sql;
    标题 "Section 15 — entities with at least one sanctioned officer";
    选择 entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   shiti_fengxian
    条件  sanctioned_officer = 1
    order 按照 risk desc;
quit;

## 16. 通道型 vs 沉淀型司法管辖区分类

**参考文献：** Garcia-Bernardo J, Fichtner J, Takes F W, Heemskerk E M.
*Uncovering Offshore Financial Centres: Conduits and Sinks in the
Global Corporate Ownership Network.* Scientific Reports 7: 6246
(2017). [DOI 10.1038/s41598-017-06322-9](https://doi.org/10.1038/s41598-017-06322-9)。

Garcia-Bernardo 等人将全球企业所有权图划分为"沉淀型 OFC"
（sink-OFCs，企业价值在此终结：BM、KY、BVI、JE、IM）与"通道型 OFC"
（conduit-OFCs，价值经由此流动：NL、UK、CH、SG、IE）。天堂文件图
拥有不同的总体（主要是注册于 Appleby 的实体），但同样的结构性区分
应能把高管聚集、地址终结的司法管辖区，与那些主要充当资本渠道的
管辖区分开。

对每个司法管辖区，我们直接从实时图计算五个结构性特征：

| 特征 | 表征 |
|---|---|
| `log(n_entity)` | 该司法管辖区离岸存在的绝对规模 |
| `avg_officers` | 每实体的高管密度（沉淀型管辖区每个实体承载更多高管） |
| `avg_xborder_off` | 本国与实体司法管辖区不同的高管的平均数量（通道型指标） |
| `intermediary_share` | 具有 CONNECTED_TO 中介链接的实体占比 |
| `address_share` | 具有 REGISTERED_ADDRESS 链接的实体占比（沉淀型指标） |

我们标准化为 z 分数，然后应用 Ward 最小方差层次聚类。`PROC TREE`
绘制树状图。请注意，天堂文件中的 Intermediary 节点是通过
`CONNECTED_TO` 而非 `INTERMEDIARY_OF` 连接到 Entity 节点的 ——
后者在模式中存在，但对本次泄露不承载任何数据 —— 所以我们在此
使用 `CONNECTED_TO`。


In [ ]:
/* 从实时图中拉取按司法管辖区的结构性特征。 */
过程 gql conn=icij out=s16_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.country_codes IS NOT NULL
                  AND o.country_codes <> e.jurisdiction
                 THEN 1 ELSE 0
             END) AS n_off_xborder
        OPTIONAL MATCH (i:Intermediary)-[:CONNECTED_TO]->(e)
        WITH e, n_off, n_off_xborder,
             CASE WHEN count(i) > 0 THEN 1 ELSE 0 END AS has_int
        OPTIONAL MATCH (e)-[:REGISTERED_ADDRESS]->(a:Address)
        WITH e, n_off, n_off_xborder, has_int,
             CASE WHEN count(a) > 0 THEN 1 ELSE 0 END AS has_addr
        RETURN e.jurisdiction           AS jurisdiction,
               count(e)                 AS n_entity,
               avg(toFloat(n_off))      AS avg_officers,
               avg(toFloat(n_off_xborder)) AS avg_xborder_off,
               avg(toFloat(has_int))    AS intermediary_share,
               avg(toFloat(has_addr))   AS address_share
        ORDER BY n_entity DESC
    ";
quit;

/* 只保留至少有十个实体的司法管辖区，使标准化 z 分数有意义。
   天堂文件泄露共有 44 个司法管辖区；其中 23 个满足此阈值。 */
数据 s16_filtered;
    设置 s16_raw;
    如果 n_entity >= 10;
    log_n_entity = log(n_entity);
运行;

过程 均值 数据=s16_filtered noprint;
    变量 log_n_entity avg_officers avg_xborder_off
        intermediary_share address_share;
    输出 out=s16_stats
        mean=m1 m2 m3 m4 m5
        std=s1 s2 s3 s4 s5;
运行;

数据 s16_std;
    如果 _n_ = 1 那么 设置 s16_stats;
    设置 s16_filtered;
    z1 = (log_n_entity       - m1) / max(0.0001, s1);
    z2 = (avg_officers       - m2) / max(0.0001, s2);
    z3 = (avg_xborder_off    - m3) / max(0.0001, s3);
    z4 = (intermediary_share - m4) / max(0.0001, s4);
    z5 = (address_share      - m5) / max(0.0001, s5);
    保留 jurisdiction z1 z2 z3 z4 z5;
运行;

过程 打印 数据=s16_std;
    标题 "Section 16 — standardised jurisdiction features";
运行;

/* Ward's minimum-variance hierarchical clustering. */
过程 cluster 数据=s16_std method=ward outtree=s16_tree;
    变量 z1 z2 z3 z4 z5;
    id jurisdiction;
    标题 "Section 16 — Ward clustering (Garcia-Bernardo 2017 typology)";
运行;

/* Render the dendrogram. */
过程 tree 数据=s16_tree horizontal;
    id jurisdiction;
    标题 "Section 16 — conduit-vs-sink dendrogram, Paradise Papers";
运行;

## 17. 高管网络角色的主成分

**参考文献：** Borgatti S P, Everett M G. *Models of core/periphery
structures.* Social Networks 21, 375-395 (2000)。
[DOI 10.1016/S0378-8733(99)00019-2](https://doi.org/10.1016/S0378-8733(99)00019-2)。
另见 Newman M E J, *Networks: An Introduction*（Oxford, 2010），
第 7 章。

天堂文件图中的高管扮演着结构上截然不同的角色。有些位于一大簇
相关实体的中心；有些把原本相互分离的簇连接起来（即 Burt/Borgatti
意义上的中介经纪人）；少数则构成某个司法管辖区内部人网络的密集
核心。四个图度量刻画了这些不同角色：

| 度量 | 刻画 |
|---|---|
| `degree` | `OFFICER_OF` 出边的数量 —— 关联的广度 |
| `betweenness` | 经过该高管的最短路径的比例 —— **经纪中介性** |
| `kcore` | 该高管所处的 k-连通子图中最大的 k —— **核心嵌入度** |
| `pagerank` | 来自同一投影的类特征向量得分 —— **借由有影响力伙伴而产生的影响力** |

我们通过 Neo4j Graph Data Science 库在完整的
`(Officer)—[OFFICER_OF]—(Entity)` 无向投影上计算全部四个度量，
限制在度数最高的 5,000 名高管上，并对经过对数变换的度量运行
`PROC PRINCOMP`。

PCA 把四个相关的度量压缩为正交轴，其相对载荷告诉我们哪些角色在
经验上聚在一起、哪些在结构上彼此分离。

*关于局部聚类系数的说明：* Garcia-Bernardo 等人把局部聚类系数
作为第五个度量纳入。天堂文件的 `(Officer)—[:OFFICER_OF]—(Entity)`
投影严格是二部的，因此不可能存在三角形 —— 每个局部聚类系数都是 0。
我们将其从度量集合中剔除。


In [ ]:
/* PROC NETWORK 通过只读 MATCH 拉取一个前 5000 名高管的子图，
   并进程内计算度数、特征向量中心性和 k-core。它取代了此前的
   gds.graph.project + 四次 gds.<algorithm>.stream 调用 —— 那种
   模式需要一个 GDS 写模式的投影步骤，而平台只读的 step-neo4j
   服务会拒绝它。

   刻意省略了介数中心性：NetworkX 以 O(V·E) 计算精确介数，
   在这个子图上（5,000 名高管 × 约 78,000 条边）会主导运行时间。
   PCA 仍保留三个正交轴 —— 度数（局部显著性）、特征向量中心性
   （全局影响力）和 k-core（结构内聚性）—— 足以为核心解读分离
   高管角色。 */
过程 network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id,
                        top.name    AS a_name,
                        e.node_id   AS b_node_id"
    direction = undirected
    outNodes  = s17_metrics_full;
    linksvar from=a_node_id 到=b_node_id;
    centrality degree eigen=unweight;
    core;
运行;

/* 拉取用于过滤的高管节点 id/名称。 */
过程 gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer)
           RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

/* 过滤到高管行。特征向量中心性替代 PageRank —— 见第 4 小节
   的说明。 */
过程 sql;
    create table s17_metrics as
    选择 n.node                          as node_id,
           o.officer_name                  as officer_name,
           n.centr_degree                  as degree,
           coalesce(n.core_out, 0)         as kcore,
           coalesce(n.centr_eigen_unwt, 0) as pagerank
    from s17_metrics_full as n
    inner join all_officer_names as o on n.node = o.node_id
    order 按照 n.centr_degree desc;
quit;

数据 s17_metrics;
    设置 s17_metrics;
    log_degree = log(degree + 1);
    log_pr     = log(pagerank * 1000 + 1);
    k_val      = kcore;
    保留 node_id officer_name degree pagerank kcore
         log_degree log_pr k_val;
运行;

过程 打印 数据=s17_metrics (obs=10);
    标题 "Section 17 — top-10 officers by degree (raw + log metrics)";
运行;

过程 均值 数据=s17_metrics n mean std min max;
    变量 log_degree log_pr k_val;
    标题 "Section 17 — log-transformed metric summary";
运行;

过程 princomp 数据=s17_metrics out=s17_scores;
    变量 log_degree log_pr k_val;
    标题 "Section 17 — PCA of officer network roles";
运行;

过程 sgplot 数据=s17_scores;
    scatter x=prin1 y=prin2 / markerattrs=(symbol=circle size=4);
    xaxis 标签="PC1 (global influence axis)";
    yaxis 标签="PC2 (brokerage vs core embeddedness)";
    标题 "Section 17 — officers in 2D principal-component role space";
运行;

## 18. 对成立率的 ARIMA 干预分析

**参考文献：** Box G E P, Tiao G C. *Intervention analysis with
applications to economic and environmental problems.* Journal of the
American Statistical Association 70(349): 70-79 (1975)。
[DOI 10.1080/01621459.1975.10480264](https://doi.org/10.1080/01621459.1975.10480264)。
其在离岸泄露上的应用见 Koeppl T, Sipiczki I, Zhao H, *FinCEN
Files: Regulatory response and compliance*（2021）。

天堂文件图中新设实体的年度数量，是一条从 1970 年（36 个实体）到
2007 年（1,595 个实体，峰值）近乎单调增长的序列，随后是 2008-2009
年的下滑，以及到 2014 年（泄露覆盖的终点）较缓慢的高原。

我们应用经典的 Box-Tiao 干预分析，来检验两个真实世界事件是否在
成立序列中留下了可度量的印记：

- **2009 年阶跃** —— G20 伦敦峰会对避税天堂的打击（2009 年 4 月），
  这与金融危机同期发生。
- **2014 年阶跃** —— 美国 FATCA（外国账户税收合规法案）于
  2014 年 7 月 1 日生效。

响应变量取 `log(n)`，因此 -0.30 的干预系数大致对应年度成立率下降
约 26%。我们拟合 `ARIMA(1,0,0)` —— 即对这条强相关序列的 AR(1)
自回归模型 —— 并把两个阶跃虚拟变量作为外生 `INPUT=` 变量。

原假设是：一旦考虑了 AR(1) 趋势，两个阶跃都不会产生可度量的
偏移。已发表的方法学对如何解释"不拒绝"存在分歧：它既可能意味着
干预没有效果，也可能意味着 AR(1) 自相关吸收了干预信号。


In [ ]:
过程 gql conn=icij out=year_counts;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year
        RETURN year, count(*) AS n
        ORDER BY year
    ";
quit;

/* 构建干预序列数据集。两个阶跃虚拟变量：
   step_2009 = I{year >= 2009} 捕获 G20 伦敦/FATCA 预告阶段的
   制度变化；step_2014 = I{year >= 2014} 捕获 FATCA 生效日的冲击。
   响应变量为 log(n)，因此系数估计可解读为比例效应。 */
数据 s18_ts;
    设置 year_counts;
    log_n     = log(n);
    step_2009 = (year >= 2009);
    step_2014 = (year >= 2014);
    trend     = year - 1992;
运行;

过程 打印 数据=s18_ts;
    标题 "Section 18 — annual new-entity formations + intervention dummies";
运行;

过程 sgplot 数据=s18_ts;
    series x=year y=n / markers;
    refline 2009 / axis=x 标签="G20 2009"
                   lineattrs=(color=red pattern=shortdash);
    refline 2014 / axis=x 标签="FATCA 2014"
                   lineattrs=(color=blue pattern=shortdash);
    xaxis 标签="Incorporation year";
    yaxis 标签="New entities per year";
    标题 "Section 18 — Paradise-Papers annual formations, 1970-2014";
运行;

/* 先识别模型，再用两个阶跃输入估计 ARIMA(1,0,0)。PROC ARIMA 的
   CROSSCORR= 在 IDENTIFY 步注册外生变量，以便它们可供
   ESTIMATE INPUT= 使用。 */
过程 arima 数据=s18_ts;
    identify 变量=log_n crosscorr=(step_2009 step_2014) nlag=8;
    估计 p=1 输入=(step_2009 step_2014);
    标题 "Section 18 — ARIMA(1,0,0) with G20-2009 and FATCA-2014 steps";
运行;
quit;

## 19. 零膨胀的制裁暴露计数模型

**参考文献：** Cameron A C, Trivedi P K. *Regression Analysis of Count
Data*, 第 2 版, Cambridge University Press (2013), 第 4 章。
零膨胀模型：Lambert D. *Zero-inflated Poisson regression
with an application to defects in manufacturing.* Technometrics
34(1): 1-14 (1992)。
[DOI 10.2307/1269547](https://doi.org/10.2307/1269547)。

第 14 小节仅发现**五个**天堂文件实体拥有至少一名列于合并制裁名单
上的高管 —— 这是一个极其罕见的事件。对每个实体的 `sanctioned_count`
做标准的 Poisson 或负二项回归会拟合不当，因为 **99.98%** 的实体
取值为零。零膨胀负二项（ZINB）模型通过把拟合拆成两部分来处理
这一点：

1. 一个逻辑模型，判断实体是否属于"结构性零"类别（不可能有制裁
   暴露）。
2. 一个负二项模型，对其余实体中的计数进行建模。

在 21,538 个实体中仅有 5 个正事件的情况下，ZINB 似然是退化的 ——
两部分都会坍缩。这一失败是**数据的诚实属性**，而非过程的问题。
我们仍然运行 ZINB 拟合，以证明回归工具链能端到端工作，然后退回到
对 `has_sanctioned`（`sanctioned_count > 0` 的指示变量）的普通二元
逻辑回归。该逻辑回归识别出一个清晰的效应：**高管数量每增加一个
对数单位，拥有至少一名被制裁高管的几率就乘以约 3.1**（p = 0.002）。

协变量：

- `top5` —— 6 水平的类变量（BM、KY、VG、IM、JE、OTHER），
  参照类别为 OTHER。
- `log_n_off` —— `log(officer_count)`，主导性的规模预测因子。


In [ ]:
/* 从实时图中拉取按实体的被制裁高管计数。 */
过程 gql conn=icij out=s19_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.node_id IN [
                     '80081590','80105707','80061592'
                 ] THEN 1 ELSE 0 END) AS n_sanc
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               e.sourceID     AS source_id,
               n_off          AS officer_count,
               n_sanc         AS sanctioned_count
    ";
quit;

数据 s19;
    设置 s19_raw;
    如果 officer_count >= 1;
    长度 top5 $5;
    top5 = 'OTHER';
    如果 jurisdiction = 'BM' 那么 top5 = 'BM';
    否则 如果 jurisdiction = 'KY' 那么 top5 = 'KY';
    否则 如果 jurisdiction = 'VG' 那么 top5 = 'VG';
    否则 如果 jurisdiction = 'IM' 那么 top5 = 'IM';
    否则 如果 jurisdiction = 'JE' 那么 top5 = 'JE';
    log_n_off      = log(officer_count);
    has_sanctioned = (sanctioned_count > 0);
运行;

过程 频率 数据=s19;
    tables top5 sanctioned_count has_sanctioned;
    标题 "Section 19 — response-variable distribution (very zero-heavy)";
运行;

/* 先做 ZINB —— 预期在仅有 5 个事件的序列上会退化。 */
过程 genmod 数据=s19;
    分类 top5;
    模型 sanctioned_count = top5 log_n_off / dist=zinb link=log;
    标题 "Section 19 — ZINB count model (degenerate on 5 events)";
运行;

/* 回退方案：对 has_sanctioned 做二元逻辑回归。可解释。 */
过程 logistic 数据=s19 descending plots=none;
    分类 top5;
    模型 has_sanctioned = top5 log_n_off;
    标题 "Section 19 — logistic regression (has-sanctioned fallback)";
运行;

## 20. 混合效应 Poisson 成立率模型

**参考文献：** McCulloch C E, Searle S R. *Generalized, Linear, and
Mixed Models*. Wiley (2001)。面板计数经典文献：Hausman J A,
Hall B H, Griliches Z. *Econometric models for count data with an
application to the patents-R&D relationship.* Econometrica 52(4):
909-938 (1984)。
[DOI 10.2307/1911191](https://doi.org/10.2307/1911191)。

第 18 小节对聚合的成立序列拟合了一元 ARIMA。这里我们利用**面板**
维度：每个司法管辖区-年份单元一行，拟合一个 Poisson 广义线性
混合模型（GLMM），带有固定的线性年份趋势加上一个 FATCA 阶跃
虚拟变量，以及**按司法管辖区的随机截距**。这把共同趋势效应与
各司法管辖区自身的水平分离开来。

面板限制：在 1990-2014 年间**平均年度计数**为 >=5 的 10 个司法
管辖区（共 203 个单元）。计数为零的年份很多的较小管辖区会使
Poisson 拟合不稳定。

`PROC GLIMMIX` 配合 `DIST=POISSON LINK=LOG` 和
`RANDOM INTERCEPT / SUBJECT=jurisdiction`，同时产出总体层面的固定
效应（年份趋势、FATCA 偏移）与司法管辖区间的方差分量。随机截距
方差告诉我们*在考虑了共同时间趋势之后，各司法管辖区的成立率相差
多少* —— 这是离岸金融中心总体的一个结构性异质性得分。


In [ ]:
/* 面板数据集：1990-2014 年的司法管辖区 × 年份单元。 */
过程 gql conn=icij out=s20_raw;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1990
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* 保留平均年度计数 >= 5 的司法管辖区。 */
过程 sql;
    create table s20_keep_jur as
    选择 jurisdiction, avg(n) as avg_n
    from s20_raw
    group 按照 jurisdiction
    having avg(n) >= 5;
quit;

过程 sql;
    create table s20 as
    选择 r.*,
           r.year - 2002 as year_c,
           case 当条件 r.year >= 2014 那么 1 否则 0 结束 as fatca
    from s20_raw r
    inner join s20_keep_jur k
    on r.jurisdiction = k.jurisdiction;
quit;

过程 频率 数据=s20;
    tables jurisdiction;
    标题 "Section 20 — guanxiaqu retained in the panel";
运行;

/* 混合效应 Poisson GLMM：固定的年份趋势 + FATCA 阶跃，
   按司法管辖区的随机截距。 */
过程 glimmix 数据=s20;
    分类 jurisdiction;
    模型 n = year_c fatca / dist=poisson link=log solution;
    random intercept / subject=jurisdiction;
    标题 "Section 20 — mixed Poisson formation-rate model";
运行;

/* 对随机截距效应排序会呈现出那些系统性地跑赢或跑输共同趋势的
   司法管辖区。PROC GLIMMIX 的 SOLUTION 语句已在上面的默认输出中
   打印这些内容 —— 排序工作留给读者。 */

In [ ]:
/* 关闭报告 PDF 并释放 Neo4j 库引用。 */
ods pdf close;

libname icij clear;

## 可复现性与溯源

- **图数据源：** ICIJ *Offshore Leaks* 研究数据库，*天堂文件*
  数据集，首次发布于 2017 年 11 月。可在
  <https://offshoreleaks.icij.org/pages/database> 获取。
  通过上游公开转储 `github.com/neo4j-graph-examples/icij-paradise-papers`
  载入平台共享的 `step-neo4j` 服务
  （Neo4j 5.26 社区版，服务器层级只读）。
- **OFAC SDN 数据：** 美国财政部 OFAC 特别指定国民公开 CSV 导出，
  于 2026 年 4 月从财政部公开预览 API 获取。`data/ofac_sdn.csv`
  文件包含按条目数量排名前五的 OFAC 项目中精选的 500 行。用于
  第 6 小节的两阶段筛查。
- **OpenSanctions 数据：** OpenSanctions *default* 合并数据集
  2026-04-17 的快照，从
  `https://data.opensanctions.org/datasets/20260417/default/targets.simple.csv`
  下载。提交的文件 `data/opensanctions_default.csv` 包含 18,654 条
  Person 与 Company 模式的行，它们的主数据集属于 OFAC SDN、英国
  财政部、欧盟金融制裁、联合国安理会，或加拿大、比利时、澳大利亚、
  瑞士等国家的合并制裁名单之一。为把文件控制在 2 MB 以内，别名
  已被剔除。许可证：ODbL 1.0（OpenSanctions）。用于第 14 小节的
  富化。
- **避税天堂排名：** 税收正义网络 *2021 年企业避税天堂指数*
  发布的排名，整理自 `https://cthi.taxjustice.net` 指数着陆页与
  2021 年 3 月的新闻稿
  `https://taxjustice.net/press/tax-haven-ranking-shows-countries-setting-global-tax-rules-do-most-to-help-firms-bend-them/`。
  提交的文件 `data/tax_haven_ranks.csv` 列出了出现在天堂文件中的
  司法管辖区及其 CTHI 排名和派生的 `[0, 1]` 风险权重。许可证：
  CC BY-SA 4.0（税收正义网络）。用于第 15 小节的综合排名。
- **图算法：** Louvain 社区检测与特征向量中心性（最接近 PageRank
  的自有替代方案）由 `PROC NETWORK` 在通过只读 Cypher 拉取的边列表
  上进程内计算。平台共享的 `step-neo4j` 服务在服务器层级只读，
  因此所有图算法都在 Workspace pod 中运行，而非通过 Neo4j Graph
  Data Science 的写操作。
- **统计方法：** `PROC LIFETEST` 使用 Kaplan-Meier 估计量，配合
  log-rank、Wilcoxon 和 Tarone-Ware 分层检验。`PROC PHREG` 通过
  Breslow 结点法使用 Python/statsmodels 包装器拟合 Cox 比例风险
  模型。`PROC LOGISTIC` 拟合极大似然二元逻辑回归。
- **风险合成方法：** 第 11 小节的综合影响力得分把度数、log-PageRank、
  司法管辖区广度和任期年数归一化到 `[0, 1]`，并以固定权重
  （30/30/20/20）求和。第 15 小节的综合实体风险得分把封顶的高管
  数量、log-PageRank、CTHI 风险权重，以及二元的被制裁高管标记
  归一化，并以各 0.25 的相等权重求和。

完整分析可由本文件夹中的冒烟测试脚本 `./smoke.jenner` 复现。
将其针对共享的 `step-neo4j` 服务端到端运行（并设置好
`JENNER_NEO4J_HOST` 和 `JENNER_NEO4J_PASS`，平台会在任何 Workspace
pod 中为你完成设置）大约需要两分钟，并会验证每个查询和每个 PROC ——
包括在既有流水线之外新增的五个小节 —— 在真实的 163,435 节点图上
都返回预期输出。
